In [100]:
import duckdb
import pandas as pd

%load_ext sql

%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', '{:f}'.format)
pd.set_option('display.width', None)

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [101]:
conn = duckdb.connect('../data/analytics.duckdb')
%sql conn --alias duckdb

## 1. Prescriber Activity & Specialties

Examines prescribing volume, leading prescribers, specialty-level differences, and concentration of claims.


### Prescriber Volume


In [102]:
%%sql
-- What are the name, specialty, and total claim count of the prescriber with the highest claim volume?
SELECT
    nppes_provider_first_name,
    nppes_provider_last_org_name,
    specialty_description,
    SUM(total_claim_count)
FROM prescriber
    INNER JOIN prescription USING (npi)
GROUP BY npi, nppes_provider_first_name, nppes_provider_last_org_name, specialty_description
ORDER BY SUM(total_claim_count) DESC;

,nppes_provider_first_name,nppes_provider_last_org_name,specialty_description,sum(total_claim_count)
0,BRUCE,PENDLEY,Family Practice,99707.000000
1,MICHAEL,COX,Internal Medicine,68721.000000
2,DANA,NASH,Family Practice,65659.000000
3,JEFFERY,WARREN,Family Practice,62301.000000
4,DONALD,VOLLMER,Family Practice,55499.000000
...,...,...,...,...
20587,HOLLY,MEEKER,Nurse Practitioner,11.000000
20588,FERNANDO,DOMINGO,Student in an Organized Health Care Education/Training Program,11.000000
20589,RUTH,CASTIELLO,Nurse Practitioner,11.000000
20590,MOLLY,FORD,Colorectal Surgery (Proctology),11.000000


In [103]:
%%sql
-- Who were the five highest-volume prescribers in Nashville by total claims?
SELECT
	nppes_provider_first_name,
	nppes_provider_last_org_name,
	SUM(total_claim_count)
FROM prescriber
	INNER JOIN prescription USING (npi)
WHERE nppes_provider_city LIKE 'NASHVILLE'
GROUP BY npi, nppes_provider_first_name, nppes_provider_last_org_name
ORDER BY SUM(total_claim_count) DESC
LIMIT 5;

,nppes_provider_first_name,nppes_provider_last_org_name,sum(total_claim_count)
0,JOHN,WILLIAMS,53622.000000
1,ERNEST,JOHNSON,29929.000000
2,DAVID,EWART,26013.000000
3,RICHARD,GARMAN,25511.000000
4,DAVID,HEUSINKVELD,23703.000000


In [104]:
%%sql
-- Who were the five highest-volume prescribers in Memphis by total claims?
SELECT
	nppes_provider_first_name,
	nppes_provider_last_org_name,
	SUM(total_claim_count)
FROM prescriber
	INNER JOIN prescription USING (npi)
WHERE nppes_provider_city LIKE 'MEMPHIS'
GROUP BY npi, nppes_provider_first_name, nppes_provider_last_org_name
ORDER BY SUM(total_claim_count) DESC
LIMIT 5;

,nppes_provider_first_name,nppes_provider_last_org_name,sum(total_claim_count)
0,DANA,NASH,65659.000000
1,JEFFERY,WARREN,62301.000000
2,ELBERT,HINES,40169.000000
3,ROBERT,BURNS,39491.000000
4,ANGELA,WATSON,36190.000000


In [105]:
%%sql
-- Who were the highest-volume prescribers across Nashville, Memphis, Knoxville, and Chattanooga?
SELECT 
	nppes_provider_first_name,
	nppes_provider_last_org_name,
	SUM(total_claim_count)
FROM prescriber
	INNER JOIN prescription USING (npi)
WHERE nppes_provider_city IN ('NASHVILLE', 'MEMPHIS', 'KNOXVILLE', 'CHATTANOOGA')
GROUP BY npi, nppes_provider_first_name, nppes_provider_last_org_name
ORDER BY SUM(total_claim_count) DESC
LIMIT 5;

,nppes_provider_first_name,nppes_provider_last_org_name,sum(total_claim_count)
0,DANA,NASH,65659.000000
1,JEFFERY,WARREN,62301.000000
2,JOHN,WILLIAMS,53622.000000
3,ELBERT,HINES,40169.000000
4,ROBERT,BURNS,39491.000000


### Specialty Comparisons


In [106]:
%%sql
-- Which specialty had the highest total number of claims across all drugs?
SELECT specialty_description, SUM(total_claim_count)
FROM prescriber
    INNER JOIN prescription USING (npi)
GROUP BY specialty_description
ORDER BY SUM(total_claim_count) DESC;

,specialty_description,sum(total_claim_count)
0,Family Practice,9752347.000000
1,Internal Medicine,9150489.000000
2,Nurse Practitioner,7185315.000000
3,Cardiology,1720149.000000
4,Physician Assistant,1419766.000000
...,...,...
87,Clinical Medical Laboratory,16.000000
88,Radiology,15.000000
89,"Psychologist, Clinical",11.000000
90,Thoracic Surgery (Cardiothoracic Vascular Surgery),11.000000


In [107]:
%%sql
-- Which specialty had the highest total number of opioid claims
SELECT specialty_description, SUM(total_claim_count)
FROM prescriber
	INNER JOIN prescription USING (npi)
	INNER JOIN drug USING (drug_name)
WHERE opioid_drug_flag = 'Y'
GROUP BY specialty_description
ORDER BY SUM(total_claim_count) DESC;

,specialty_description,sum(total_claim_count)
0,Nurse Practitioner,900845.000000
1,Family Practice,467246.000000
2,Internal Medicine,402114.000000
3,Physician Assistant,204084.000000
4,Orthopedic Surgery,78584.000000
...,...,...
69,Surgery,39.000000
70,Case Manager/Care Coordinator,36.000000
71,Allergy/ Immunology,12.000000
72,Critical Care (Intensivists),11.000000


In [108]:
%%sql
-- What percentage of each specialty's total claims were for opioids, and which specialties had the highest percentages?
SELECT 
    specialty_description,
	ROUND(SUM(CASE WHEN opioid_drug_flag = 'Y' THEN total_claim_count ELSE 0 END) / SUM(total_claim_count) * 100, 2) AS opioid_percentage
FROM prescription
	INNER JOIN prescriber USING (npi)
	INNER JOIN drug USING (drug_name)
GROUP BY specialty_description
ORDER BY opioid_percentage DESC;

,specialty_description,opioid_percentage
0,Case Manager/Care Coordinator,72.000000
1,Orthopaedic Surgery,68.980000
2,Interventional Pain Management,59.470000
3,Anesthesiology,58.520000
4,Pain Management,58.080000
...,...,...
87,"Psychologist, Clinical",0.000000
88,Licensed Clinical Social Worker,0.000000
89,Pharmacist,0.000000
90,Family Medicine,0.000000


### Prescriber Coverage and Missing Records


In [109]:
%%sql
-- Which specialties have no associated prescriptions?
SELECT specialty_description
FROM prescriber
EXCEPT
SELECT specialty_description
FROM prescriber
	INNER JOIN prescription USING (npi);

,specialty_description
0,Undefined Physician type
1,Hospital
2,Occupational Therapist in Private Practice
3,Radiology Practitioner Assistant
4,Midwife
5,Chiropractic
6,Developmental Therapist
7,Licensed Practical Nurse
8,Physical Therapy Assistant
9,Physical Therapist in Private Practice


In [110]:
%%sql
-- Prescribers (or NPIs) with no available prescription data
SELECT npi
FROM prescriber
EXCEPT
SELECT npi
FROM prescriber
	INNER JOIN prescription USING (npi);

,npi
0,1003097593
1,1013094028
2,1013426717
3,1023192135
4,1023472743
...,...
4453,1982994711
4454,1992094783
4455,1992167100
4456,1992860779


### Opioid Prescribing Concentration


In [111]:
%%sql
-- What proportion of opioids are prescribed by the top 10 prescribers? top 50? top 100?
WITH top_10 AS (
    SELECT
        SUM(total_claim_count) AS top_sum
    FROM prescription
        INNER JOIN prescriber USING (npi)
        INNER JOIN drug USING (drug_name)
    WHERE opioid_drug_flag = 'Y'
    GROUP BY npi
    ORDER BY SUM(total_claim_count) DESC
    LIMIT 10
), top_50 AS (
    SELECT
        SUM(total_claim_count) AS top_sum
    FROM prescription
        INNER JOIN prescriber USING (npi)
        INNER JOIN drug USING (drug_name)
    WHERE opioid_drug_flag = 'Y'
    GROUP BY npi
    ORDER BY SUM(total_claim_count) DESC
    LIMIT 50
), top_100 AS (
    SELECT
        SUM(total_claim_count) AS top_sum
    FROM prescription
        INNER JOIN prescriber USING (npi)
        INNER JOIN drug USING (drug_name)
    WHERE opioid_drug_flag = 'Y'
    GROUP BY npi
    ORDER BY SUM(total_claim_count) DESC
    LIMIT 100
)
SELECT
    100.0 * (SELECT SUM(top_sum) FROM top_10) / SUM(total_claim_count) AS top_10,
    100.0 * (SELECT SUM(top_sum) FROM top_50) / SUM(total_claim_count) AS top_50,
    100.0 * (SELECT SUM(top_sum) FROM top_100) / SUM(total_claim_count) AS top_100
FROM prescription
    INNER JOIN prescriber USING (npi)
    INNER JOIN drug USING (drug_name)
WHERE opioid_drug_flag = 'Y';

,top_10,top_50,top_100
0,2.404669,8.369886,13.859758


### Unanswered Questions

*Who is the top prescriber in each county?*
- Prescriptions are tied to zip codes and there are multiple zip codes in some counties. Can't reliably be deduced.


## 2. Drug Utilization & Prescription Patterns

Examines drug classifications, prescription volume, and differences in drug utilization across specialties.


### Drug Classification


In [112]:
%%sql
-- Which drugs were classified as opioids, antibiotics, or neither?
SELECT drug_name,
	CASE WHEN opioid_drug_flag = 'Y' THEN 'opioid'
		WHEN antibiotic_drug_flag = 'Y' THEN 'antiobiotic'
	ELSE 'neither' END AS drugtype
FROM drug;

,drug_name,drugtype
0,1ST TIER UNIFINE PENTIPS,neither
1,1ST TIER UNIFINE PENTIPS PLUS,neither
2,ABACAVIR,neither
3,ABACAVIR-LAMIVUDINE,neither
4,ABACAVIR-LAMIVUDINE-ZIDOVUDINE,neither
...,...,...
3420,ZYPREXA,neither
3421,ZYPREXA RELPREVV,neither
3422,ZYPREXA ZYDIS,neither
3423,ZYTIGA,neither


### High-Volume Prescriptions


In [113]:
%%sql
-- Which prescription records had at least 3000 total claims?
SELECT drug_name, total_claim_count
FROM prescription
WHERE total_claim_count >= 3000;

,drug_name,total_claim_count
0,OXYCODONE HCL,4538.000000
1,LEVOTHYROXINE SODIUM,3023.000000
2,HYDROCODONE-ACETAMINOPHEN,3376.000000
3,LEVOTHYROXINE SODIUM,3101.000000
4,GABAPENTIN,3531.000000
5,LEVOTHYROXINE SODIUM,3138.000000
6,MIRTAZAPINE,3085.000000
7,LISINOPRIL,3655.000000
8,FUROSEMIDE,3083.000000


In [114]:
%%sql
-- Which drugs with at least 3000 total claims were opioids?
SELECT 
    drug_name, total_claim_count
FROM prescription
    INNER JOIN drug USING (drug_name)
WHERE total_claim_count >= 3000
    AND opioid_drug_flag = 'Y';

,drug_name,total_claim_count
0,OXYCODONE HCL,4538.000000
1,HYDROCODONE-ACETAMINOPHEN,3376.000000


In [115]:
%%sql
-- Which prescribers were associated with drugs having at least 3000 total claims?
SELECT DISTINCT 
	nppes_provider_first_name, 
	nppes_provider_last_org_name
FROM prescriber
	INNER JOIN ( 
		SELECT npi 
		FROM prescription 
			INNER JOIN drug USING (drug_name) 
		WHERE total_claim_count >= 3000 AND opioid_drug_flag = 'Y'
	) AS opioid_prescribers USING (npi);

,nppes_provider_first_name,nppes_provider_last_org_name
0,DAVID,COFFEY


### Drug Utilization by Specialty


In [116]:
%%sql
-- What were the five most-prescribed generic drugs among Family Practice prescribers?
SELECT generic_name, SUM(total_claim_count)
FROM prescription
	INNER JOIN drug USING (drug_name)
	INNER JOIN prescriber USING (npi)
WHERE specialty_description LIKE 'Family Practice'
GROUP BY generic_name
ORDER BY SUM(total_claim_count) DESC
LIMIT 5;

,generic_name,sum(total_claim_count)
0,LEVOTHYROXINE SODIUM,406547.000000
1,LISINOPRIL,311506.000000
2,ATORVASTATIN CALCIUM,308523.000000
3,AMLODIPINE BESYLATE,304343.000000
4,OMEPRAZOLE,273570.000000


In [117]:
%%sql
-- What were the five most-prescribed generic drugs among Cardiology prescribers?
SELECT generic_name, SUM(total_claim_count)
FROM prescription
	INNER JOIN drug USING (drug_name)
	INNER JOIN prescriber USING (npi)
WHERE specialty_description LIKE 'Cardiology'
GROUP BY generic_name
ORDER BY SUM(total_claim_count) DESC
LIMIT 5;

,generic_name,sum(total_claim_count)
0,ATORVASTATIN CALCIUM,120662.000000
1,CARVEDILOL,106812.000000
2,METOPROLOL TARTRATE,93940.000000
3,CLOPIDOGREL BISULFATE,87025.000000
4,AMLODIPINE BESYLATE,86928.000000


In [118]:
%%sql
-- Which generic drugs ranked among the most prescribed by Family Practice and Cardiology prescribers?
SELECT generic_name, SUM(total_claim_count)
FROM prescription
	INNER JOIN drug USING (drug_name)
	INNER JOIN prescriber USING (npi)
WHERE specialty_description LIKE 'Family Practice' OR
	specialty_description LIKE 'Cardiology'
GROUP BY generic_name
ORDER BY SUM(total_claim_count) DESC
LIMIT 5;

,generic_name,sum(total_claim_count)
0,ATORVASTATIN CALCIUM,429185.000000
1,LEVOTHYROXINE SODIUM,415476.000000
2,AMLODIPINE BESYLATE,391271.000000
3,LISINOPRIL,387799.000000
4,FUROSEMIDE,318196.000000


### Pain-Management Prescribing


In [119]:
%%sql
-- What were all possible prescriber-drug combinations for Nashville pain-management specialists and opioid drugs?
SELECT npi, drug_name
FROM prescriber
	CROSS JOIN drug
WHERE specialty_description = 'Pain Management' AND
	nppes_provider_city = 'NASHVILLE' AND
	opioid_drug_flag = 'Y';

,npi,drug_name
0,1154436590,ABSTRAL
1,1154436590,ACETAMIN-CAFF-DIHYDROCODEINE
2,1154436590,ACETAMINOPHEN-CODEINE
3,1154436590,ACTIQ
4,1154436590,ARYMO ER
...,...,...
632,1952473431,XODOL 10-300
633,1952473431,XODOL 7.5-300
634,1952473431,XTAMPZA ER
635,1952473431,XYLON 10


In [120]:
%%sql
-- What was the claim count for every Nashville pain-management specialist and opioid combination (with missing claims reported as zero)?
SELECT 
    npi, 
    drug.drug_name, 
    COALESCE(SUM(total_claim_count), 0)
FROM prescriber
	CROSS JOIN drug
	LEFT JOIN prescription USING (npi, drug_name)
WHERE specialty_description = 'Pain Management' AND
	nppes_provider_city = 'NASHVILLE' AND
	opioid_drug_flag = 'Y'
GROUP BY npi, drug.drug_name
ORDER BY SUM(total_claim_count) DESC;

,npi,drug_name,"COALESCE(sum(total_claim_count), 0)"
0,1457685976,HYDROCODONE-ACETAMINOPHEN,665.000000
1,1457685976,OXYCODONE-ACETAMINOPHEN,480.000000
2,1457685976,OXYCODONE HCL,326.000000
3,1932256898,HYDROCODONE-ACETAMINOPHEN,308.000000
4,1932256898,OXYCODONE HCL,191.000000
...,...,...,...
632,1811044621,REPREXAIN,0.000000
633,1932256898,TYLENOL-CODEINE NO.4,0.000000
634,1952473431,OXYCODONE HCL-IBUPROFEN,0.000000
635,1952473431,SUBSYS,0.000000


## 3. Prescription Spending & Drug Costs

Examines prescription costs, cost efficiency by supply duration, and opioid-related spending.


### Drug Cost Comparisons


In [121]:
%%sql
-- Which generic drug had the highest total drug cost?
SELECT generic_name, SUM(total_drug_cost)
FROM drug
	INNER JOIN prescription USING (drug_name)
GROUP BY generic_name
ORDER BY SUM(total_drug_cost) DESC;

,generic_name,sum(total_drug_cost)
0,"INSULIN GLARGINE,HUM.REC.ANLOG",104264066.350000
1,PREGABALIN,78645939.890000
2,APIXABAN,73201341.080000
3,LENALIDOMIDE,70282523.770000
4,LEDIPASVIR/SOFOSBUVIR,60344875.980000
...,...,...
1194,ERGOCALCIFEROL (VITAMIN D2),212.420000
1195,CYCLOPENTOLATE HCL,175.870000
1196,"PRENATAL VIT,CAL 74/IRON/FOLIC",131.040000
1197,FOLIC ACID,120.950000


In [122]:
%%sql
-- Which generic drug had the highest total cost per day supplied?
SELECT 
    generic_name, 
    ROUND(SUM(total_drug_cost) / SUM(total_day_supply), 2) AS cost_per_day
FROM drug
	INNER JOIN prescription USING (drug_name)
GROUP BY generic_name
ORDER BY cost_per_day DESC;

,generic_name,cost_per_day
0,C1 ESTERASE INHIBITOR,3495.220000
1,ICATIBANT ACETATE,3216.770000
2,CHENODIOL,2891.370000
3,ASFOTASE ALFA,2369.340000
4,CORTICOTROPIN,2148.430000
...,...,...
1194,LISINOPRIL/HYDROCHLOROTHIAZIDE,0.130000
1195,FUROSEMIDE,0.120000
1196,HYDROCHLOROTHIAZIDE,0.100000
1197,ERGOCALCIFEROL (VITAMIN D2),0.090000


### Spending by Drug Classification


In [123]:
%%sql
-- Was more spent on opioids or antibiotics?
SELECT
	CASE WHEN opioid_drug_flag = 'Y' THEN 'opioid'
		WHEN antibiotic_drug_flag = 'Y' THEN 'antiobiotic'
		ELSE 'neither' END AS drugtype,
	SUM(total_drug_cost)::DECIMAL(18,2) AS total_spent
FROM drug
	INNER JOIN prescription USING (drug_name)
GROUP BY drugtype
ORDER BY total_spent DESC;

,drugtype,total_spent
0,neither,2972698710.230000
1,opioid,105080626.370000
2,antiobiotic,38435121.260000


### Relative Spending


In [124]:
%%sql
-- What is the ratio for spending on opioid vs non-opioid prescriptions?
SELECT
    opioid_drug_flag,
    SUM(total_drug_cost) /
        (SELECT SUM(total_drug_cost) FROM prescription INNER JOIN drug USING(drug_name)) AS spending_percent
FROM prescription
    INNER JOIN drug USING (drug_name)
GROUP BY GROUPING SETS ((), (opioid_drug_flag));
    

,opioid_drug_flag,spending_percent
0,NaN,1.000000
1,N,0.966279
2,Y,0.033721


### Unanswered Questions

*What is the correlation between spending on opioids and overdose deaths?*
- No dates on prescriptions to reliably connect drug spending with overdose deaths

*Are those spending more on opioids suffering from more deaths?*
- Death numbers are separated by county. Drug spending is separated by zip code.

*Which county has the most spending overall per capita?*..
*Which county has the most spending on opioids per capita?*
- Drug spending is tied to zip codes, but not counties.



## 4. Tennessee Geography & Population

Establishes the geographic and demographic context needed for regional comparisons.


### Population Distribution


In [125]:
%%sql
-- What was the total population of Tennessee?
SELECT SUM(population)
FROM population
	INNER JOIN fips_county USING (fipscounty)
WHERE state = 'TN';

,sum(population)
0,6597381.000000


In [126]:
%%sql
-- What percentage of Tennessee's population lived in each county?
SELECT county,
	population,
	ROUND(population / (SELECT SUM(population)
	FROM population INNER JOIN fips_county USING (fipscounty)
	WHERE state = 'TN'), 3) AS pop_percentage
FROM population
	INNER JOIN fips_county USING (fipscounty)
WHERE state = 'TN'
GROUP BY county, population
ORDER BY pop_percentage DESC;

,county,population,pop_percentage
0,SHELBY,937847.000000,0.142000
1,DAVIDSON,678322.000000,0.103000
2,KNOX,452286.000000,0.069000
3,HAMILTON,354589.000000,0.054000
4,RUTHERFORD,298456.000000,0.045000
...,...,...,...
90,VAN BUREN,5675.000000,0.001000
91,CLAY,7684.000000,0.001000
92,PERRY,7882.000000,0.001000
93,TROUSDALE,8773.000000,0.001000


### Metropolitan and Statistical Areas


In [127]:
%%sql
-- How many CBSAs are in Tennessee?
SELECT COUNT(DISTINCT cbsa)
FROM cbsa
	INNER JOIN fips_county USING (fipscounty)
WHERE state = 'TN';

,count(DISTINCT cbsa)
0,10


In [128]:
%%sql
-- Which CBSAs had the largest and smallest combined populations?
SELECT cbsaname, SUM(population) as comb_pop
FROM cbsa
	INNER JOIN population USING (fipscounty)
GROUP BY cbsaname
ORDER BY comb_pop DESC;

,cbsaname,comb_pop
0,"Nashville-Davidson--Murfreesboro--Franklin, TN",1830410.000000
1,"Memphis, TN-MS-AR",1038617.000000
2,"Knoxville, TN",862490.000000
3,"Chattanooga, TN-GA",397598.000000
4,"Kingsport-Bristol-Bristol, TN-VA",212921.000000
5,"Johnson City, TN",200767.000000
6,"Clarksville, TN-KY",192120.000000
7,"Jackson, TN",129538.000000
8,"Cleveland, TN",120388.000000
9,"Morristown, TN",116352.000000


In [129]:
%%sql
-- What was the most populous county not included in a CBSA?
SELECT county, population
FROM fips_county
	INNER JOIN population USING (fipscounty)
	LEFT JOIN cbsa USING (fipscounty)
WHERE cbsa IS NULL
ORDER BY population DESC
LIMIT 1;

,county,population
0,SEVIER,95523.000000


## 5. Opioid Claims & Regional Patterns

Focuses on opioid prescribing across specialties, cities, and individual prescribers, moving toward multidimensional comparisons.


### Opioid Prescriber Geography


In [130]:
%%sql
-- Where are the top 10 opioid prescribers located?
SELECT
    nppes_provider_last_org_name,
    nppes_provider_city,
    nppes_provider_state
FROM prescriber
    INNER JOIN (
        SELECT
            npi,
            SUM(total_claim_count)
        FROM prescription
            INNER JOIN drug USING (drug_name)
        WHERE opioid_drug_flag = 'Y'
        GROUP BY npi
        ORDER BY SUM(total_claim_count) DESC
        LIMIT 10
    ) USING (npi);

,nppes_provider_last_org_name,nppes_provider_city,nppes_provider_state
0,CATHERS,KNOXVILLE,TN
1,PAINTER,BRISTOL,TN
2,WILLETT,KINGSTON,TN
3,LADSON,MURFREESBORO,TN
4,GREEN,KNOXVILLE,TN
5,TAYLOR,LA FOLLETTE,TN
6,KINDRICK,CROSSVILLE,TN
7,COFFEY,ONEIDA,TN
8,BOWSER,GALLATIN,TN
9,CLARK,JAMESTOWN,TN


In [131]:
%%sql
-- Which zip codes in Davidson County have the most opioids prescribed?
SELECT
    nppes_provider_zip5,
    SUM(total_claim_count)
FROM prescriber
    INNER JOIN prescription USING (npi)
    INNER JOIN drug USING (drug_name)
    INNER JOIN (
        SELECT zip 
        FROM zip_fips 
        INNER JOIN fips_county USING (fipscounty) 
        WHERE county = 'DAVIDSON'
    ) AS zip_list ON prescriber.nppes_provider_zip5 = zip_list.zip
WHERE opioid_drug_flag = 'Y'
GROUP BY nppes_provider_zip5
ORDER BY SUM(total_claim_count) DESC
LIMIT 10;

,nppes_provider_zip5,sum(total_claim_count)
0,37203,83587.000000
1,37013,40306.000000
2,37232,31713.000000
3,37076,26684.000000
4,37205,24507.000000
5,37027,18707.000000
6,37207,17967.000000
7,37211,10761.000000
8,37122,7409.000000
9,37204,6213.000000


### Pain-Management Specialty Comparisons


In [132]:
%%sql
-- How many total claims were associated with Interventional Pain Management and Pain Management specialists?
SELECT specialty_description, SUM(total_claim_count) AS total_claims
FROM prescriber
	INNER JOIN prescription USING (npi)
WHERE specialty_description = 'Interventional Pain Management' OR
	specialty_description = 'Pain Management'
GROUP BY specialty_description;

,specialty_description,total_claims
0,Pain Management,70853.000000
1,Interventional Pain Management,55906.000000


In [133]:
%%sql
-- What were the specialty-level and combined claim totals for Interventional Pain Management and Pain Management specialists? 
SELECT specialty_description, SUM(total_claim_count) AS total_claims
FROM prescriber
	INNER JOIN prescription USING (npi)
	INNER JOIN drug USING (drug_name)
WHERE specialty_description = 'Interventional Pain Management' OR
	specialty_description = 'Pain Management'
GROUP BY GROUPING SETS ((),(specialty_description));

,specialty_description,total_claims
0,NaN,129726.000000
1,Pain Management,72487.000000
2,Interventional Pain Management,57239.000000


In [134]:
%%sql
-- How did opioid and non-opioid claims compare across the two pain-management specialties (including category and overall totals)?
SELECT specialty_description, opioid_drug_flag, SUM(total_claim_count) AS total_claims
FROM prescriber
	INNER JOIN prescription USING (npi)
	INNER JOIN drug USING (drug_name)
WHERE specialty_description = 'Interventional Pain Management' OR
	specialty_description = 'Pain Management'
GROUP BY GROUPING SETS ((),(opioid_drug_flag),(specialty_description));

,specialty_description,opioid_drug_flag,total_claims
0,NaN,NaN,129726.000000
1,NaN,N,53583.000000
2,NaN,Y,76143.000000
3,Pain Management,NaN,72487.000000
4,Interventional Pain Management,NaN,57239.000000


### Multidimensional Claim Analysis


In [135]:
%%sql
-- How did claim totals break down by opioid status and then by pain-management specialty?
SELECT specialty_description, opioid_drug_flag, SUM(total_claim_count) AS total_claims
FROM prescriber
	INNER JOIN prescription USING (npi)
	INNER JOIN drug USING (drug_name)
WHERE specialty_description = 'Interventional Pain Management' OR
	specialty_description = 'Pain Management'
GROUP BY ROLLUP(opioid_drug_flag, specialty_description);

,specialty_description,opioid_drug_flag,total_claims
0,NaN,NaN,129726.000000
1,NaN,N,53583.000000
2,NaN,Y,76143.000000
3,Pain Management,N,30386.000000
4,Interventional Pain Management,N,23197.000000
5,Pain Management,Y,42101.000000
6,Interventional Pain Management,Y,34042.000000


In [136]:
%%sql
-- How did claim totals break down by pain-management specialty and then by opioid status?
SELECT specialty_description, opioid_drug_flag, SUM(total_claim_count) AS total_claims
FROM prescriber
	INNER JOIN prescription USING (npi)
	INNER JOIN drug USING (drug_name)
WHERE specialty_description = 'Interventional Pain Management' OR
	specialty_description = 'Pain Management'
GROUP BY ROLLUP(specialty_description, opioid_drug_flag);

,specialty_description,opioid_drug_flag,total_claims
0,NaN,NaN,129726.000000
1,Interventional Pain Management,NaN,57239.000000
2,Pain Management,NaN,72487.000000
3,Interventional Pain Management,Y,34042.000000
4,Pain Management,N,30386.000000
5,Interventional Pain Management,N,23197.000000
6,Pain Management,Y,42101.000000


In [137]:
%%sql
-- What were the claim totals across every combination of pain-management specialty and opioid status?
SELECT specialty_description, opioid_drug_flag, SUM(total_claim_count) AS total_claims
FROM prescriber
	INNER JOIN prescription USING (npi)
	INNER JOIN drug USING (drug_name)
WHERE specialty_description = 'Interventional Pain Management' OR
	specialty_description = 'Pain Management'
GROUP BY CUBE(specialty_description, opioid_drug_flag);

,specialty_description,opioid_drug_flag,total_claims
0,NaN,NaN,129726.000000
1,Interventional Pain Management,NaN,57239.000000
2,Pain Management,NaN,72487.000000
3,Interventional Pain Management,N,23197.000000
4,Pain Management,N,30386.000000
5,Pain Management,Y,42101.000000
6,Interventional Pain Management,Y,34042.000000
7,NaN,N,53583.000000
8,NaN,Y,76143.000000


### Cross-City Drug Comparison


In [138]:
%%sql
-- How many claims were recorded for six common opioid categories across Nashville, Memphis, Knoxville, and Chattanooga?
PIVOT (
    SELECT
        nppes_provider_city AS city,
        CASE
            WHEN drug_name ILIKE '%hydrocodone%' THEN 'hydrocodone'
            WHEN drug_name ILIKE '%oxycodone%' THEN 'oxycodone'
            WHEN drug_name ILIKE '%oxymorphone%' THEN 'oxymorphone'
            WHEN drug_name ILIKE '%morphine%' THEN 'morphine'
            WHEN drug_name ILIKE '%codeine%' THEN 'codeine'
            WHEN drug_name ILIKE '%fentanyl%' THEN 'fentanyl'
        END AS drug_category,
        total_claim_count
    FROM prescription
    INNER JOIN prescriber USING (npi)
    WHERE nppes_provider_city IN (
        'NASHVILLE', 'MEMPHIS', 'KNOXVILLE', 'CHATTANOOGA'
    )
    AND drug_category IS NOT NULL
)
ON drug_category
USING SUM(total_claim_count)
GROUP BY city
ORDER BY city;

,city,codeine,fentanyl,hydrocodone,morphine,oxycodone,oxymorphone
0,CHATTANOOGA,1310.000000,3689.000000,68256.000000,11076.000000,46522.000000,798.000000
1,KNOXVILLE,2717.000000,4811.000000,78359.000000,18461.000000,79965.000000,4703.000000
2,MEMPHIS,4697.000000,3655.000000,67951.000000,4133.000000,36948.000000,63.000000
3,NASHVILLE,2020.000000,6119.000000,88572.000000,11676.000000,59149.000000,738.000000


## 6. Overdose Deaths & Population-Adjusted Outcomes

Examines overdose mortality across Tennessee counties, including county comparisons, rates, and changes over time.


### County-Level Overdose Deaths


In [139]:
%%sql
SELECT county, SUM(overdose_deaths)
FROM fips_county
	INNER JOIN overdose_deaths ON overdose_deaths.fipscounty = fips_county.fipscounty::numeric
GROUP BY county
HAVING SUM(overdose_deaths) > (SELECT AVG(overdose_deaths) FROM overdose_deaths)
ORDER BY SUM(overdose_deaths) DESC;

,county,sum(overdose_deaths)
0,DAVIDSON,689.000000
1,KNOX,683.000000
2,SHELBY,567.000000
3,RUTHERFORD,205.000000
4,HAMILTON,191.000000
5,SULLIVAN,131.000000
6,MONTGOMERY,101.000000
7,SUMNER,100.000000
8,BLOUNT,99.000000
9,WILSON,98.000000


### Population-Adjusted Measures


In [140]:
%%sql
-- Which county has the highest overdose deaths per capita?
SELECT
    county,
    SUM(overdose_deaths) / MAX(population) AS deaths_capita
FROM overdose_deaths
    INNER JOIN fips_county ON fips_county.fipscounty::INTEGER = overdose_deaths.fipscounty
    INNER JOIN population ON population.fipscounty::INTEGER = overdose_deaths.fipscounty
WHERE state = 'TN'
GROUP BY county
ORDER BY deaths_capita DESC;

,county,deaths_capita
0,CHEATHAM,0.001838
1,HANCOCK,0.001514
2,KNOX,0.001510
3,ROANE,0.001455
4,CLAY,0.001432
...,...,...
90,CARROLL,0.000142
91,PERRY,0.000127
92,HAYWOOD,0.000111
93,DYER,0.000053


### Overdose Deaths over Time


In [141]:
%%sql
-- How has total overdose deaths changed over time?
SELECT
    year,
    SUM(overdose_deaths)
FROM overdose_deaths
    INNER JOIN fips_county ON fips_county.fipscounty::INTEGER = overdose_deaths.fipscounty
WHERE state = 'TN'
GROUP BY year;

,year,sum(overdose_deaths)
0,2015,1033.000000
1,2016,1186.000000
2,2017,1267.000000
3,2018,1304.000000


In [142]:
%%sql
-- How have deaths changed over time for Davidson and Shelby?
PIVOT (
    SELECT overdose_deaths, year, county
    FROM overdose_deaths
        INNER JOIN fips_county ON fips_county.fipscounty::INTEGER = overdose_deaths.fipscounty
    WHERE county IN ('DAVIDSON', 'SHELBY')
    )
ON year
USING SUM(overdose_deaths)
GROUP BY county;

,county,2015,2016,2017,2018
0,DAVIDSON,127.000000,178.000000,184.000000,200.000000
1,SHELBY,135.000000,150.000000,159.000000,123.000000


### Trend Estimation


In [143]:
%%sql
-- Are there any counties in which overdose deaths are trending downward?
SELECT
    county,
    REGR_SLOPE(overdose_deaths, year) AS trend_slope
FROM overdose_deaths
    INNER JOIN fips_county ON fips_county.fipscounty::INTEGER = overdose_deaths.fipscounty
WHERE year BETWEEN 2015 AND 2018
GROUP BY county
HAVING REGR_SLOPE(overdose_deaths, year) < 0
ORDER BY trend_slope;

,county,trend_slope
0,CAMPBELL,-3.800000
1,WASHINGTON,-3.000000
2,DICKSON,-2.900000
3,SHELBY,-2.700000
4,MC NAIRY,-2.700000
5,MORGAN,-2.400000
6,HAWKINS,-2.300000
7,WILSON,-2.200000
8,HAMBLEN,-2.200000
9,WHITE,-1.700000


## DuckDB Disconnect

In [144]:
conn.close()